# 03 — Modèle d'attribution des zones à rénover

**Rôle : DOT / voirie d'une collectivité territoriale.**

> *« Sur quelles zones concentrer nos prochaines rénovations, et quel élément y traiter en priorité ? »*

**Pipeline en 2 temps** (unité d'analyse = la **cellule H3**, équi-surface) :

1. **Sélection structurelle par le volume** — ranking des zones par **charge d'accidents pondérée par la gravité** `charge = Σ severity`. Un fort volume au même endroit = problème *structurel* ; un accident grave isolé = *erreur humaine*, hors levier DOT.
2. **Attribution** — pour chaque zone retenue, quel élément d'infrastructure est responsable, via **deux méthodes qui doivent converger** :
   - **2A — sur-représentation** (lift de l'élément vs référence nationale, pondéré gravité) ;
   - **2B — modèle de fréquence + SHAP** (charge ~ composition infra, explication locale par zone).

**Pondération gravité (transversale)** : chaque accident pèse sa sévérité (1→4).
**Périmètre des éléments = infrastructure uniquement** (on répond à une question de rénovation structurelle ; météo/heure = hors scope).
**Limite assumée** : sans données d'exposition (OSM/trafic), l'univers du modèle = cellules ayant déjà ≥ 1 accident → on explique *« parmi les lieux accidentogènes, quelle composition fait monter la charge »* (cohérent avec l'étape 1).

In [1]:
# Dépendances (à exécuter une seule fois)
%pip install -q h3 folium branca scipy kagglehub pyspark scikit-learn shap


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: /Users/saba/Prog/Epita/ING2/xPloring/.venv/bin/python3.12 -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import folium
import branca.colormap as cmap
import h3

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import shap

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, BooleanType, TimestampType,
)

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('attribution_zones_ca')
    .config('spark.driver.memory', '6g')
    .config('spark.sql.session.timeZone', 'UTC')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('Spark', spark.version, '| h3', h3.__version__, '| shap', shap.__version__)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/08 17:18:42 WARN Utils: Your hostname, MacBook-Pro-de-Tristan.local, resolves to a loopback address: 127.0.0.1; using 10.90.133.253 instead (on interface en0)
26/06/08 17:18:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/06/08 17:18:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.1.1 | h3 4.4.2 | shap 0.52.0


In [3]:
# Chargement reproductible : CSV local si présent, sinon téléchargement KaggleHub
LOCAL_CANDIDATES = [
    'US_Accidents_March23.csv',
    '../US_Accidents_March23.csv',
    os.path.expanduser('~/Prog/Epita/ING2/xPloring/US_Accidents_March23.csv'),
]
CSV_PATH = next((p for p in LOCAL_CANDIDATES if os.path.exists(p)), None)

if CSV_PATH is None:
    import kagglehub
    path = kagglehub.dataset_download('sobhanmoosavi/us-accidents')
    csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
    assert csv_files, 'Aucun CSV trouvé dans le répertoire KaggleHub.'
    CSV_PATH = os.path.join(path, csv_files[0])

print('CSV utilisé :', CSV_PATH)

CSV utilisé : /Users/saba/Prog/Epita/ING2/xPloring/US_Accidents_March23.csv


In [4]:
# Schéma explicite (évite la double passe inferSchema)
schema = StructType([
    StructField('ID', StringType()),
    StructField('Source', StringType()),
    StructField('Severity', IntegerType()),
    StructField('Start_Time', TimestampType()),
    StructField('End_Time', TimestampType()),
    StructField('Start_Lat', DoubleType()),
    StructField('Start_Lng', DoubleType()),
    StructField('End_Lat', DoubleType()),
    StructField('End_Lng', DoubleType()),
    StructField('Distance(mi)', DoubleType()),
    StructField('Description', StringType()),
    StructField('Street', StringType()),
    StructField('City', StringType()),
    StructField('County', StringType()),
    StructField('State', StringType()),
    StructField('Zipcode', StringType()),
    StructField('Country', StringType()),
    StructField('Timezone', StringType()),
    StructField('Airport_Code', StringType()),
    StructField('Weather_Timestamp', TimestampType()),
    StructField('Temperature(F)', DoubleType()),
    StructField('Wind_Chill(F)', DoubleType()),
    StructField('Humidity(%)', DoubleType()),
    StructField('Pressure(in)', DoubleType()),
    StructField('Visibility(mi)', DoubleType()),
    StructField('Wind_Direction', StringType()),
    StructField('Wind_Speed(mph)', DoubleType()),
    StructField('Precipitation(in)', DoubleType()),
    StructField('Weather_Condition', StringType()),
    StructField('Amenity', BooleanType()),
    StructField('Bump', BooleanType()),
    StructField('Crossing', BooleanType()),
    StructField('Give_Way', BooleanType()),
    StructField('Junction', BooleanType()),
    StructField('No_Exit', BooleanType()),
    StructField('Railway', BooleanType()),
    StructField('Roundabout', BooleanType()),
    StructField('Station', BooleanType()),
    StructField('Stop', BooleanType()),
    StructField('Traffic_Calming', BooleanType()),
    StructField('Traffic_Signal', BooleanType()),
    StructField('Turning_Loop', BooleanType()),
    StructField('Sunrise_Sunset', StringType()),
    StructField('Civil_Twilight', StringType()),
    StructField('Nautical_Twilight', StringType()),
    StructField('Astronomical_Twilight', StringType()),
])

df = spark.read.option('header', True).schema(schema).csv(CSV_PATH)
df = df.toDF(*[c.lower().replace('(', '_').replace(')', '').replace('%', 'pct') for c in df.columns])
print('Colonnes infra disponibles :', [c for c in df.columns if c in
      ['junction','crossing','stop','traffic_signal','railway','station',
       'roundabout','bump','give_way','no_exit','traffic_calming','amenity']])

Colonnes infra disponibles : ['amenity', 'bump', 'crossing', 'give_way', 'junction', 'no_exit', 'railway', 'roundabout', 'station', 'stop', 'traffic_calming', 'traffic_signal']


## Étape 0 — Agrégation au niveau zone (Californie)

On filtre la **Californie** (1,74 M accidents, démonstration ; le modèle est générique et entraînable sur tout le US), on encode chaque accident dans sa cellule **H3 résolution 8** (~0,7 km², l'échelle d'un carrefour), puis on agrège **par zone** avec la pondération gravité.

In [5]:
INFRA = ['junction', 'crossing', 'stop', 'traffic_signal', 'railway', 'station',
         'roundabout', 'bump', 'give_way', 'no_exit', 'traffic_calming', 'amenity']

df_ca = (
    df.filter(F.col('state') == 'CA')
      .select('start_lat', 'start_lng', 'severity', *INFRA)
      .dropna(subset=['start_lat', 'start_lng', 'severity'])
)
df_pd = df_ca.toPandas()
for c in INFRA:
    df_pd[c] = df_pd[c].fillna(False).astype(int)
print(f'Californie : {len(df_pd):,} accidents × {df_pd.shape[1]} colonnes')

Californie : 1,741,433 accidents × 15 colonnes


In [6]:
H3_RES = 8
# Compatibilité h3-py v3 / v4
try:
    h3.latlng_to_cell(0.0, 0.0, H3_RES)
    h3_encode   = lambda lat, lng: h3.latlng_to_cell(lat, lng, H3_RES)
    h3_boundary = lambda cell: h3.cell_to_boundary(cell)
    h3_center   = lambda cell: h3.cell_to_latlng(cell)
except AttributeError:
    h3_encode   = lambda lat, lng: h3.geo_to_h3(lat, lng, H3_RES)
    h3_boundary = lambda cell: h3.h3_to_geo_boundary(cell)
    h3_center   = lambda cell: h3.h3_to_geo(cell)

df_pd['h3_cell'] = [h3_encode(lat, lng)
                    for lat, lng in zip(df_pd['start_lat'], df_pd['start_lng'])]
print(f'{df_pd["h3_cell"].nunique():,} cellules H3 — densité moy. '
      f'{len(df_pd)/df_pd["h3_cell"].nunique():.1f} accidents/cellule')

58,857 cellules H3 — densité moy. 29.6 accidents/cellule


In [7]:
# Agrégation par zone, avec pondération gravité (vectorisé).
#   charge       = somme des sévérités  (volume pondéré gravité)
#   charge_<el>  = somme des sévérités des accidents touchant l'élément
#   n_<el>       = nombre d'accidents touchant l'élément
for el in INFRA:                      # severity x indicatrice (pré-calcul vectorisé)
    df_pd['sev_' + el] = df_pd['severity'] * df_pd[el]

aggmap = {'severity': ['count', 'mean', 'sum']}
for el in INFRA:
    aggmap[el]         = 'sum'        # -> n_<el>
    aggmap['sev_' + el] = 'sum'       # -> charge_<el>

zone = df_pd.groupby('h3_cell').agg(aggmap)
zone.columns = (['n_accidents', 'avg_severity', 'charge']
                + [c for el in INFRA for c in ('n_' + el, 'charge_' + el)])
zone = zone.reset_index()
print(f'Zones agrégées : {len(zone):,}')
zone[['n_accidents', 'avg_severity', 'charge']].describe().round(2)

Zones agrégées : 58,857


,n_accidents,avg_severity,charge
count,58857.00,58857.00,58857.00
mean,29.59,2.04,64.08
std,107.38,0.17,244.07
min,1.00,1.00,1.00
25%,2.00,2.00,4.00
50%,4.00,2.00,8.00
75%,12.00,2.00,24.00
max,2861.00,4.00,7236.00


## Étape 1 — Sélection structurelle par la charge pondérée

On classe les zones par `charge` et on retient le **top-N** comme zones prioritaires à rénover. (Cohérent avec le `risk_score = n_accidents × avg_severity` de `02_points_noirs.ipynb`, ici exprimé directement comme somme des sévérités.)

In [8]:
PRIORITY_N = 50
zone = zone.sort_values('charge', ascending=False).reset_index(drop=True)
zone['rank'] = zone.index + 1
priority = zone.head(PRIORITY_N).copy()

print(f'TOP {PRIORITY_N} ZONES PRIORITAIRES — par charge pondérée gravité\n' + '=' * 55)
disp = priority[['rank', 'n_accidents', 'avg_severity', 'charge']].head(10).copy()
disp.columns = ['Rang', 'Accidents', 'Sév. moy.', 'Charge (Σ sév.)']
disp['Sév. moy.'] = disp['Sév. moy.'].round(3)
disp

TOP 50 ZONES PRIORITAIRES — par charge pondérée gravité


,Rang,Accidents,Sév. moy.,Charge (Σ sév.)
0,1,2861,2.529,7236
1,2,2543,2.497,6350
2,3,2431,2.327,5658
3,4,2174,2.576,5601
4,5,2127,2.543,5409
5,6,2163,2.472,5348
6,7,2201,2.418,5321
7,8,2323,2.049,4760
8,9,2085,2.258,4707
9,10,2099,2.234,4689


## Étape 2A — Attribution par sur-représentation (lift)

Pour chaque zone et chaque élément :

$$\text{lift}(el, zone) = \frac{\text{part pondérée de } el \text{ dans la zone}}{\text{part pondérée de } el \text{ au niveau national (CA)}}$$

où la *part pondérée* = `charge_<el> / charge`. Un lift > 1 signifie que l'élément concentre **plus** de charge d'accidents ici qu'en moyenne. L'élément au lift maximal = **suspect dominant**. Méthode descriptive, robuste, lisible par un décideur.

In [9]:
# Référence nationale (Californie) : part pondérée de chaque élément
charge_tot = zone['charge'].sum()
nat_share = {el: zone['charge_' + el].sum() / charge_tot for el in INFRA}
print('Part nationale (CA) de la charge par élément :')
for el, v in sorted(nat_share.items(), key=lambda x: -x[1]):
    print(f'  {el:18s} {v*100:5.1f}%')

Part nationale (CA) de la charge par élément :
  junction            10.9%
  traffic_signal       7.4%
  crossing             5.1%
  stop                 3.0%
  station              2.3%
  railway              1.1%
  amenity              0.7%
  give_way             0.1%
  no_exit              0.1%
  traffic_calming      0.1%
  bump                 0.1%
  roundabout           0.0%


In [10]:
# Seuil : on n'attribue un élément que s'il est réellement SUR-représenté.
# Sinon (aucun lift > seuil) la zone n'a pas de signal infra dominant -> 'diffus'.
LIFT_MIN = 1.5

def lift_row(r):
    shares = {el: (r['charge_' + el] / r['charge']) / nat_share[el]
              if nat_share[el] > 0 else 0.0 for el in INFRA}
    top = max(shares, key=shares.get)
    if shares[top] < LIFT_MIN:
        return pd.Series({'dominant_A': 'diffus', 'lift_A': shares[top]})
    return pd.Series({'dominant_A': top, 'lift_A': shares[top]})

priority[['dominant_A', 'lift_A']] = priority.apply(lift_row, axis=1)
n_diffus = (priority['dominant_A'] == 'diffus').sum()
print(f"Élément dominant (2A) par zone — distribution "
      f"({n_diffus}/{len(priority)} sans signal infra dominant) :")
print(priority['dominant_A'].value_counts())

Élément dominant (2A) par zone — distribution (13/50 sans signal infra dominant) :
dominant_A
diffus             13
railway             9
junction            6
bump                6
station             6
no_exit             3
amenity             2
traffic_signal      2
stop                1
traffic_calming     1
crossing            1
Name: count, dtype: int64


## Étape 2B — Modèle de fréquence + explication locale SHAP

On entraîne un modèle qui prédit la **charge** d'une zone à partir de sa **composition d'infrastructure** (`n_<el>`). On utilise **toutes** les zones (pas seulement les points noirs) pour que le modèle apprenne le lien composition → charge sur l'ensemble du territoire.

Puis, pour chaque zone prioritaire, les **valeurs SHAP locales** décomposent la charge prédite : l'élément à la plus forte contribution positive est le **driver** de cette zone.

> Modèle : `RandomForestRegressor` (robuste, importances interprétables, compatible `shap.TreeExplainer`). GBT était le plan initial ; RF est retenu pour la compatibilité SHAP native et la rapidité sur données agrégées.

In [11]:
FEATURES = ['n_' + el for el in INFRA]
X = zone[FEATURES].values
y = zone['charge'].values

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor(n_estimators=200, max_depth=None,
                              min_samples_leaf=5, n_jobs=-1, random_state=42)
model.fit(X_tr, y_tr)

pred = model.predict(X_te)
print('Évaluation (held-out 20%) :')
print(f'  R²   = {r2_score(y_te, pred):.3f}')
print(f'  MAE  = {mean_absolute_error(y_te, pred):.2f}')
print(f'  RMSE = {np.sqrt(mean_squared_error(y_te, pred)):.2f}')

Évaluation (held-out 20%) :
  R²   = 0.673
  MAE  = 40.50
  RMSE = 136.50


In [12]:
# Importance globale des features
imp = pd.Series(model.feature_importances_, index=INFRA).sort_values(ascending=False)
print('Importance globale des éléments (modèle 2B) :')
for el, v in imp.items():
    print(f'  {el:18s} {v*100:5.1f}%')

Importance globale des éléments (modèle 2B) :
  junction            70.4%
  traffic_signal      20.4%
  station              2.8%
  crossing             2.8%
  stop                 1.7%
  amenity              0.8%
  no_exit              0.5%
  railway              0.4%
  traffic_calming      0.1%
  bump                 0.1%
  give_way             0.0%
  roundabout           0.0%


In [13]:
# SHAP local sur les zones prioritaires
explainer = shap.TreeExplainer(model)
shap_prio = explainer.shap_values(priority[FEATURES].values)

# Driver = élément à la plus forte contribution SHAP positive
drv_idx = shap_prio.argmax(axis=1)
priority['driver_B'] = [INFRA[i] for i in drv_idx]
priority['shap_B']   = shap_prio[np.arange(len(priority)), drv_idx]
print('Driver SHAP (2B) par zone — distribution :')
print(priority['driver_B'].value_counts())

Driver SHAP (2B) par zone — distribution :
driver_B
junction          43
traffic_signal     7
Name: count, dtype: int64


## Étape 3 — Quel aménagement manque, et quel impact ? (prescription par lieu)

On ne veut pas un diagnostic général (« junction partout ») mais, **pour chaque zone**, l'aménagement concret à réaliser. La logique :

1. **Effet conditionnel à l'aléa.** Pour chaque aléa local H, on mesure quelle mesure réduit la sévérité *là où H est présent* : `delta(P|H) = sévérité(H=1, P absent) − sévérité(H=1, P présent)`. On reste sur les **leviers réalistes du DOT** (feux, passage piéton, stop, ralentisseurs) — pas le rond-point (reconstruction lourde, rare).
2. **Par zone** : aléa local (2A) → la mesure la plus efficace *pour cet aléa* qui **manque** localement.
3. **Impact estimé** = `|delta(P|H)| × volume de la zone` = réduction de charge attendue. Local par construction : dépend de l'aléa, des manques propres à la zone, et de son volume.

> Une reco fondée sur le delta *global* recommanderait le même aménagement partout (artefact). Le conditionnement à l'aléa produit des recommandations **variées et adaptées** au type de lieu.

> Les deux lentilles précédentes nourrissent ce diagnostic : **2A** dit l'aléa local sur-représenté, **2B** confirme que junction/signal portent la charge globale. La décision d'aménagement, elle, se prend **zone par zone**.

In [14]:
# (a) Effet marginal de chaque élément sur la sévérité (delta avec - sans), sur la CA
deltas = {el: df_pd.loc[df_pd[el] == 1, 'severity'].mean()
              - df_pd.loc[df_pd[el] == 0, 'severity'].mean()
          for el in INFRA}
print('Effet marginal sur la sévérité (delta = avec - sans) :')
for el, v in sorted(deltas.items(), key=lambda x: x[1]):
    print(f'  {el:18s} {v:+.3f}   {"protecteur" if v < 0 else "AGGRAVANT"}')

Effet marginal sur la sévérité (delta = avec - sans) :
  roundabout         -0.166   protecteur
  give_way           -0.138   protecteur
  amenity            -0.122   protecteur
  stop               -0.114   protecteur
  crossing           -0.104   protecteur
  traffic_calming    -0.092   protecteur
  traffic_signal     -0.083   protecteur
  bump               -0.062   protecteur
  no_exit            -0.031   protecteur
  station            -0.025   protecteur
  junction           +0.053   AGGRAVANT
  railway            +0.061   AGGRAVANT


In [15]:
# (b) Bénéfice CONDITIONNEL d'une mesure P là où l'aléa H est présent :
#     delta(P|H) = sévérité(H=1, P absent) - sévérité(H=1, P présent)   (< 0 => P aide)
# Leviers réalistes du DOT (on exclut rond-point/voie ferrée... non « installables » à la volée)
DEPLOYABLE = ['traffic_signal', 'crossing', 'stop', 'traffic_calming']
MIN_N = 200      # échantillon mini pour un couple (H, P) fiable

def cond_delta(H, P):
    sub = df_pd[df_pd[H] == 1]
    a = sub[sub[P] == 0]['severity']
    b = sub[sub[P] == 1]['severity']
    if len(a) < MIN_N or len(b) < MIN_N:
        return np.nan
    return b.mean() - a.mean()

cond = {H: {P: cond_delta(H, P) for P in DEPLOYABLE if P != H} for H in INFRA}

print('Mesure la plus réductrice par aléa (delta conditionnel) :')
for H in INFRA:
    opts = {P: v for P, v in cond[H].items() if pd.notna(v) and v < 0}
    if opts:
        best = min(opts, key=opts.get)
        print(f'  aléa {H:16s} -> {best:16s} ({opts[best]:+.3f})')

Mesure la plus réductrice par aléa (delta conditionnel) :
  aléa junction         -> crossing         (-0.054)
  aléa crossing         -> traffic_calming  (-0.060)
  aléa stop             -> traffic_calming  (-0.052)
  aléa traffic_signal   -> traffic_calming  (-0.090)
  aléa railway          -> crossing         (-0.368)
  aléa station          -> traffic_signal   (-0.149)
  aléa bump             -> crossing         (-0.143)
  aléa give_way         -> crossing         (-0.023)
  aléa no_exit          -> crossing         (-0.153)
  aléa traffic_calming  -> crossing         (-0.117)
  aléa amenity          -> crossing         (-0.034)


In [16]:
# Moteur de recommandation par zone : aléa local -> mesure conditionnelle manquante
PRESENCE_MIN = 0.05   # élément "présent" si >= 5% des accidents de la zone le portent
LABELS = {
    'traffic_signal':  'installer des feux',
    'crossing':        'sécuriser passage piéton',
    'stop':            'stop / cédez-le-passage',
    'traffic_calming': 'modération de trafic (ralentisseurs)',
}

def recommend(r):
    present = {el: r['n_' + el] / r['n_accidents'] for el in INFRA}
    H = r['dominant_A']
    if H == 'diffus':
        return pd.Series({'amenagement': 'investiguer facteur humain (vitesse / contrôle)',
                          'impact_charge': 0.0})
    # mesures efficaces POUR CET aléa (delta conditionnel < 0), classées, qui MANQUENT localement
    ranked = sorted([(P, cond[H][P]) for P in DEPLOYABLE
                     if P != H and pd.notna(cond[H].get(P)) and cond[H][P] < 0],
                    key=lambda x: x[1])
    missing = [(P, v) for P, v in ranked if present[P] < PRESENCE_MIN]
    if not missing:
        return pd.Series({'amenagement': 'déjà équipé — revoir géométrie / abords',
                          'impact_charge': 0.0})
    P, v = missing[0]
    label = LABELS[P]
    if H == 'railway':                 # contexte ferroviaire : libellé adapté
        label = 'sécuriser le passage à niveau (barrières/feux)'
    return pd.Series({'amenagement': label, 'impact_charge': abs(v) * r['n_accidents']})

priority[['amenagement', 'impact_charge']] = priority.apply(recommend, axis=1)
print('Aménagements recommandés (top 50 zones) :')
print(priority['amenagement'].value_counts())

Aménagements recommandés (top 50 zones) :
amenagement
investiguer facteur humain (vitesse / contrôle)    13
sécuriser passage piéton                           12
sécuriser le passage à niveau (barrières/feux)      9
stop / cédez-le-passage                             6
modération de trafic (ralentisseurs)                4
installer des feux                                  4
déjà équipé — revoir géométrie / abords             2
Name: count, dtype: int64


## Livrable — recommandations d'aménagement par zone

Pour chaque zone prioritaire : volume, charge, aléa local dominant (2A), **équipement déjà présent**, **aménagement manquant recommandé**, et **impact estimé** (réduction de charge attendue). Trié par impact pour donner l'ordre d'intervention. Exporté en CSV + carte colorée par type d'aménagement.

In [17]:
def present_list(r):
    p = [el for el in INFRA if r['n_' + el] / r['n_accidents'] >= PRESENCE_MIN]
    return ', '.join(p) if p else '—'

reco = priority.copy()
reco['equip_present'] = reco.apply(present_list, axis=1)
reco = reco.sort_values('impact_charge', ascending=False)

out = reco[['rank', 'h3_cell', 'n_accidents', 'charge', 'avg_severity',
            'dominant_A', 'lift_A', 'equip_present', 'amenagement', 'impact_charge']].copy()
out['avg_severity']  = out['avg_severity'].round(3)
out['lift_A']        = out['lift_A'].round(2)
out['impact_charge'] = out['impact_charge'].round(0)
out.columns = ['Rang', 'Zone H3', 'Accidents', 'Charge', 'Sév. moy.',
               'Aléa local (2A)', 'Lift', 'Équipement présent',
               'Aménagement recommandé', 'Impact estimé']
out.to_csv('recos_amenagement_ca.csv', index=False)
print('Export : recos_amenagement_ca.csv —', len(out), 'zones')
out.head(15)

Export : recos_amenagement_ca.csv — 50 zones


,Rang,Zone H3,Accidents,Charge,Sév. moy.,Aléa local (2A),Lift,Équipement présent,Aménagement recommandé,Impact estimé
4,5,8829a56c55fffff,2127,5409,2.543,railway,30.38,"junction, railway, station",sécuriser le passage à niveau (barrières/feux),782.0
17,18,8829a1d121fffff,1676,3975,2.372,railway,2.55,"junction, traffic_signal",sécuriser le passage à niveau (barrières/feux),616.0
18,19,8829a56f37fffff,1543,3931,2.548,railway,69.81,"railway, station",sécuriser le passage à niveau (barrières/feux),567.0
27,28,8829a56c51fffff,1526,3591,2.353,railway,39.69,"junction, traffic_signal, railway, station",sécuriser le passage à niveau (barrières/feux),561.0
35,36,8829a569d3fffff,1376,3341,2.428,railway,19.00,"junction, traffic_signal, railway, station",sécuriser le passage à niveau (barrières/feux),506.0
44,45,882830813dfffff,1356,3181,2.346,railway,22.34,"junction, railway",sécuriser le passage à niveau (barrières/feux),499.0
39,40,8829a56999fffff,1329,3267,2.458,railway,3.68,traffic_signal,sécuriser le passage à niveau (barrières/feux),489.0
29,30,8829a1d2b1fffff,1600,3542,2.214,railway,5.38,"junction, crossing, traffic_signal, railway",sécuriser le passage à niveau (barrières/feux),365.0
37,38,8829a56c61fffff,1415,3304,2.335,railway,25.73,"crossing, traffic_signal, railway, station",sécuriser le passage à niveau (barrières/feux),323.0
20,21,8829a0aec7fffff,1932,3911,2.024,no_exit,50.44,"junction, no_exit",sécuriser passage piéton,295.0


In [18]:
# Carte : hexagones (détail au zoom) + pastilles en pixels (visibles dézoomé),
# taille proportionnelle à la charge, couleur = aménagement recommandé.
actions = sorted(reco['amenagement'].unique())
palette = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6', '#f39c12',
           '#1abc9c', '#e67e22', '#34495e', '#fd79a8', '#00b894']
color_of = {a: palette[i % len(palette)] for i, a in enumerate(actions)}

m = folium.Map(tiles='CartoDB positron')
charge_max = reco['charge'].max()
centers = []
for _, r in reco.iterrows():
    try:
        c = list(h3_center(r['h3_cell']))      # (lat, lng)
        centers.append(c)
        col = color_of[r['amenagement']]
        tip = (f"<b>#{int(r['rank'])}</b> — {int(r['n_accidents']):,} accidents (charge {r['charge']:.0f})<br>"
               f"Aléa local : <b>{r['dominant_A']}</b><br>"
               f"Présent : {r['equip_present']}<br>"
               f"➜ Aménagement : <b>{r['amenagement']}</b><br>"
               f"Impact estimé : {r['impact_charge']:.0f}")
        # hexagone H3 (apparait au zoom)
        b = h3_boundary(r['h3_cell'])
        folium.Polygon(list(b) + [b[0]], color=col, weight=1,
                       fill=True, fill_color=col, fill_opacity=0.30).add_to(m)
        # pastille : rayon en pixels (constant à l'écran) -> repérable même très dézoomé
        radius = 5 + 18 * (r['charge'] / charge_max)
        folium.CircleMarker(location=c, radius=radius, color='#222', weight=1,
                            fill=True, fill_color=col, fill_opacity=0.9,
                            tooltip=folium.Tooltip(tip)).add_to(m)
    except Exception:
        pass

if centers:
    m.fit_bounds(centers)                      # cadre automatiquement sur les zones prioritaires

items = ''.join(f"<div><span style='background:{color_of[a]};width:12px;height:12px;"
                f"display:inline-block;margin-right:6px;'></span>{a}</div>" for a in actions)
legend = (f"<div style='position:fixed;bottom:30px;left:30px;z-index:9999;background:white;"
          f"padding:10px;border:1px solid #999;font-size:11px;max-width:300px;'>"
          f"<b>Aménagement recommandé</b><br><i>taille = charge d'accidents</i>{items}</div>")
m.get_root().html.add_child(folium.Element(legend))

OUT_MAP = 'recos_amenagement_ca.html'
m.save(OUT_MAP)
print('Carte sauvegardée :', OUT_MAP)
m

Carte sauvegardée : recos_amenagement_ca.html


## Limites assumées (« deuils »)

- **Pas d'exposition externe.** Sans réseau routier OSM ni trafic (AADT), l'univers du modèle = cellules ayant déjà ≥ 1 accident. On explique *« parmi les lieux accidentogènes, quelle composition fait monter la charge »*, pas *« junction vs carrefour parfaitement sûr »*. Cohérent avec l'étape 1 (on ne cible que les zones à fort volume).
- **Corrélationnel, pas causal.** Les lifts et contributions SHAP sont des associations. Aucun panel avant/après ne permet de prouver qu'équiper l'élément réduira les accidents — résultats à présenter comme **signaux de priorisation**.
- **Biais de reporting** (MapQuest/Bing) : couverture inégale selon les zones.
- **Périmètre infra** : météo/heure exclus volontairement (rénovation structurelle). Le risque conjoncturel relèverait d'un livrable « mesures dynamiques » distinct.

**Extension future** : intégrer OSM pour un vrai taux d'accidents par exposition → attribution causale plus robuste.